# Data discovery and retrieval

One of the core use cases of SDMX is enabling data consumers to **discover and retrieve available datasets from a source**. This functionality is formalized in the SDMX-REST "Data Discovery and Retrieval" profile, implemented by the pysdmx connector. Below, we describe the features supported by this connector.

> **Important**
>
> To use the pysdmx connectors, you need to install the `pysdmx[data]` extra.
>
> Check the [installation guide](../../start.rst#installation) for more information.

## Setup: Initializing the pysdmx Connector

`pysdmx` offers two connectors compliant with the SDMX-REST "Data Discovery and Retrieval" profile:

- `pysdmx.api.dc.rest.SdmxConnector`: A basic connector for SDMX-REST services.
- `pysdmx.api.dc.pd.PandasConnector`: A connector, based on the one above, that returns data as a Pandas DataFrame.

For this demo, we will use the **Pandas Connector**, which can be initialized as follows:

In [11]:
from pysdmx.api.dc import Endpoints
from pysdmx.api.dc.pd import PandasConnector

conn = PandasConnector(Endpoints.BIS)

The connector requires the service's entry point URL to retrieve data and metadata, and we can easily get it using the `Endpoints` enum. Now that the connector is set up, let's explore its capabilities!

## Data Discovery, step 1: Discover Available Datasets

To begin, we need to **list available datasets** in the source to **identify those of interest**. This can be achieved using the `dataflows` method provided by the connector:

In [12]:
flows = conn.dataflows()
print(f"Found {len(flows)} dataflows.")
for f in flows:
    print(f.short_urn)

Found 32 dataflows.
Dataflow=BIS:BIS_REL_CAL(1.0)
Dataflow=BIS:WS_CBPOL(1.0)
Dataflow=BIS:WS_CBS_PUB(1.0)
Dataflow=BIS:WS_CBTA(1.0)
Dataflow=BIS:WS_CPMI_CASHLESS(1.0)
Dataflow=BIS:WS_CPMI_CT1(1.0)
Dataflow=BIS:WS_CPMI_CT2(1.0)
Dataflow=BIS:WS_CPMI_DEVICES(1.0)
Dataflow=BIS:WS_CPMI_INSTITUT(1.0)
Dataflow=BIS:WS_CPMI_MACRO(1.0)
Dataflow=BIS:WS_CPMI_PARTICIP(1.0)
Dataflow=BIS:WS_CPMI_SYSTEMS(1.0)
Dataflow=BIS:WS_CPP(1.0)
Dataflow=BIS:WS_CREDIT_GAP(1.0)
Dataflow=BIS:WS_DEBT_SEC2_PUB(1.0)
Dataflow=BIS:WS_DER_OTC_TOV(1.0)
Dataflow=BIS:WS_DPP(1.0)
Dataflow=BIS:WS_DSR(1.0)
Dataflow=BIS:WS_EER(1.0)
Dataflow=BIS:WS_GLI(1.0)
Dataflow=BIS:WS_LBS_D_PUB(1.0)
Dataflow=BIS:WS_LONG_CPI(1.0)
Dataflow=BIS:WS_NA_SEC_C3(1.0)
Dataflow=BIS:WS_NA_SEC_DSS(1.0)
Dataflow=BIS:WS_OTC_DERIV2(1.0)
Dataflow=BIS:WS_SPP(1.0)
Dataflow=BIS:WS_TC(2.0)
Dataflow=BIS:WS_XRU(1.0)
Dataflow=BIS:WS_XTD_DERIV(1.0)
Dataflow=BIS.CBS:CBS(1.0)
Dataflow=BIS.LBS:LBSN(1.0)
Dataflow=BIS.LBS:LBSR(1.0)


You can also filter datasets by providing a search term. The method will return dataflows where the term matches the ID, name, or description:

In [13]:
flows = conn.dataflows("banking")
print(f"Found {len(flows)} dataflows.")
for f in flows:
    print(f.short_urn)

Found 5 dataflows.
Dataflow=BIS:WS_CBS_PUB(1.0)
Dataflow=BIS:WS_LBS_D_PUB(1.0)
Dataflow=BIS.CBS:CBS(1.0)
Dataflow=BIS.LBS:LBSN(1.0)
Dataflow=BIS.LBS:LBSR(1.0)


## Data Discovery, step 2: Gather Information about a Dataset

After identifying a dataflow of interest, the next step is to gather detailed **information about it** to learn how to **query it effectively**. This can be achieved using the `dataflow` method provided by the connector:

### Basic information

You can print details about the dataflow, such as its name and availability metrics, if provided by the source:

In [14]:
cbs = conn.dataflow(flows[0])

print(f"Name: {cbs.name}")
print(f"Number of observations: {cbs.obs_count}")
print(f"Number of series: {cbs.series_count}")

Name: Consolidated banking
Number of observations: None
Number of series: 229388


These metrics are useful to estimate how large a query result could be. If the source does not provide them, they may be missing.

### Querying dimensions

To query data effectively, examine the dimensions and their possible values:

In [15]:
for d in cbs.components.dimensions:
    dv = [c.id for c in d.enumeration]
    print(f"{d.id}: {','.join(dv)}.")

FREQ: Q.
L_MEASURE: S,B.
L_REP_CTY: 1C,1E,1W,2B,2C,2D,2H,2N,2O,2R,2S,2T,2U,2W,2Z,3C,3P,4B,4C,4M,4N,4O,4R,4T,4U,4W,4Y,5A,5C,5J,5K,5M,5R,AD,AE,AF,AG,AI,AL,AM,AN,AO,AR,AT,AU,AW,AZ,BA,BB,BD,BE,BF,BG,BH,BI,BJ,BM,BN,BO,BQ,BR,BS,BT,BW,BY,BZ,C9,CA,CD,CF,CG,CH,CI,CL,CM,CN,CO,CR,CS,CU,CV,CW,CY,CZ,DD,DE,DJ,DK,DM,DO,DZ,EC,EE,EG,ER,ES,ET,FI,FJ,FK,FM,FO,FR,GA,GB,GD,GE,GG,GH,GI,GL,GM,GN,GQ,GR,GT,GW,GY,HK,HN,HR,HT,HU,ID,IE,IL,IM,IN,IQ,IR,IS,IT,JE,JM,JO,JP,KE,KG,KH,KI,KM,KN,KP,KR,KW,KY,KZ,LA,LB,LC,LI,LK,LR,LS,LT,LU,LV,LY,MA,MD,ME,MG,MH,MK,ML,MM,MN,MO,MR,MS,MT,MU,MV,MW,MX,MY,MZ,NA,NC,NE,NG,NI,NL,NO,NP,NR,NZ,OM,PA,PE,PF,PG,PH,PK,PL,PS,PT,PU,PW,PY,QA,RO,RS,RU,RW,SA,SB,SC,SD,SE,SG,SH,SI,SK,SL,SM,SN,SO,SR,SS,ST,SU,SV,SX,SY,SZ,TC,TD,TF,TG,TH,TJ,TL,TM,TN,TO,TR,TT,TV,TW,TZ,UA,UG,US,UY,UZ,VA,VC,VE,VG,VN,VU,WF,WS,YE,YU,ZA,ZM,ZW.
CBS_BANK_TYPE: 1C,1E,1W,2B,2C,2D,2H,2N,2O,2R,2S,2T,2U,2W,2Z,3C,3P,4B,4C,4M,4N,4O,4R,4T,4U,4W,4Y,5A,5C,5J,5K,5M,5R,AD,AE,AF,AG,AI,AL,AM,AN,AO,AR,AT,AU,AW,AZ,BA,BB,BD,BE,BF,BG,BH,BI,BJ,BM,

Rather than returning all theoretically possible values from the codelist for each dimension, the connector only provides values for which data actually exist, reducing the risk of executing queries that yield no results.

### Filtering availability information

When exploring a large dataflow, you may want to focus on a specific subset of the data first. You can do this by supplying `filters` to the `dataflow` method.

With `filters=None` (the default), the connector returns availability information for the full dataflow. With filters set, it returns availability information for the matching subset only (that is, the values that remain available once filters are applied).

For example, to inspect only Swiss cross-border claims:

In [16]:
from pysdmx.api.dc.query import MultiFilter, Operator, TextFilter

f1 = TextFilter("L_POSITION", Operator.EQUALS, "D")
f2 = TextFilter("L_REP_CTY", Operator.EQUALS, "CH")
mf = MultiFilter([f1, f2])

cbs_subset = conn.dataflow(cbs, mf)

print(f"Full flow observations: {cbs.obs_count}")
print(f"Full flow series: {cbs.series_count}")
print(f"Subset observations: {cbs_subset.obs_count}")
print(f"Subset series: {cbs_subset.series_count}")

for d in cbs_subset.components.dimensions:
    dv = [c.id for c in d.enumeration]
    print(f"{d.id}: {','.join(dv)}.")

Full flow observations: None
Full flow series: 229388
Subset observations: None
Subset series: 2
FREQ: Q.
L_MEASURE: S.
L_REP_CTY: 3P,4B,4R,5J,CH.
CBS_BANK_TYPE: 3P,4B,4R,5J,CH.
CBS_BASIS: U.
L_POSITION: D.
L_INSTR: A.
REM_MATURITY: A.
CURR_TYPE_BOOK: TO1.
L_CP_SECTOR: A.
L_CP_COUNTRY: 3P,4B,4R,5J,CH.


The filter syntax is the same as for `data` queries: you can pass query filter objects (as above), SQL-like strings, or Python boolean expressions.

## Data Retrieval: Download Data

Once you know how to query the data, the next step is to **download it** as a **Pandas Data Frame**. This can be achieved using the `data` method provided by the connector. The `data` method has one mandatory parameter, `dataflow`.

### Specifying the Dataflow

The `dataflow` parameter specifies the dataflow from which to retrieve data. It can accept the following types of input:

- **String**: A string representation of the dataflow, which can be an **SDMX URN** (e.g. `urn:sdmx:org.sdmx.infomodel.datastructure.Dataflow=BIS:CBS(1.0)`) or a short notation for the dataflow, such as `BIS:CBS(1.0)`.
- **Python Object**: A Python object that has the following three properties:
  - `id`: The unique identifier of the dataflow.
  - `agency`: The agency responsible for the dataflow.
  - `version`: The version of the dataflow.

These three properties (`id`, `agency`, and `version`) are the standard attributes used in SDMX to uniquely identify artefacts of a certain type, such as a Dataflow. Examples of Python objects that can be passed include `pysdmx.Dataflow`, `pysdmx.DataflowInfo`, or `pysdmx.Reference`.

### Applying Query Filters

To filter the data, you can use `pysdmx` query filters. For example, to retrieve cross-border claims (`L_POSITION=D`) for Switzerland (`L_REP_CTY=CH`):

In [17]:
from pysdmx.api.dc.query import MultiFilter, Operator, TextFilter

f1 = TextFilter("L_POSITION", Operator.EQUALS, "D")
f2 = TextFilter("L_REP_CTY", Operator.EQUALS, "CH")
mf = MultiFilter([f1, f2])

df = conn.data(cbs, mf)

print(df)

                                       FREQ L_MEASURE L_REP_CTY CBS_BANK_TYPE  \
SERIES_KEY                 TIME_PERIOD                                          
Q.S.CH.4B.U.D.A.A.TO1.A.3P 2005-Q2        Q         S        CH            4B   
                           2005-Q3        Q         S        CH            4B   
                           2005-Q4        Q         S        CH            4B   
                           2006-Q1        Q         S        CH            4B   
                           2006-Q2        Q         S        CH            4B   
...                                     ...       ...       ...           ...   
Q.S.CH.4R.U.D.A.A.TO1.A.5J 2025-Q2        Q         S        CH            4R   
                           2025-Q3        Q         S        CH            4R   
                           2025-Q4        Q         S        CH            4R   
                           2026-Q1        Q         S        CH            4R   
                           2

By default, the connector infers the index using the series keys and the time period. This behavior can be disabled by setting `infer_index=False`. Similarly, the generation of series keys is enabled by default and can be turned off by setting `infer_series_keys=False`.

### Alternative Query Formats

The same query can also be expressed as:

- **SQL string**: `L_POSITION = 'D' AND L_REP_CTY = 'CH'`
- **Python boolean expression**: `L_POSITION == 'D' and L_REP_CTY == 'CH'`

In [18]:
df = conn.data(cbs, "L_POSITION = 'D' AND L_REP_CTY = 'CH'")

### Data Types in the Pandas DataFrame

By default, the Pandas DataFrame applies data types based on the Data Structure Definition (DSD). You can disable this feature by setting `apply_schema=False` when calling the `data` method.

In [19]:
print(df.dtypes)

FREQ              category
L_MEASURE         category
L_REP_CTY         category
CBS_BANK_TYPE     category
CBS_BASIS         category
L_POSITION        category
L_INSTR           category
REM_MATURITY      category
CURR_TYPE_BOOK    category
L_CP_SECTOR       category
L_CP_COUNTRY      category
OBS_VALUE           string
DECIMALS          category
UNIT_MEASURE      category
UNIT_MULT         category
TIME_FORMAT       category
COLLECTION        category
ORG_VISIBILITY    category
AVAILABILITY      category
TITLE_GRP           string
OBS_STATUS        category
OBS_CONF          category
OBS_PRE_BREAK       string
dtype: object


Coded components will be defined as categorical data, making them efficient for analysis.

### Label Options for Category Fields

The `labels` parameter allows you to control how category fields are represented in the DataFrame. You can choose from the following options:

- **id** (Default): The DataFrame will include only the code IDs for category fields. This is the most compact representation.
- **name**: The code IDs are replaced with their corresponding names, as defined in the codelists.
- **both**: Both the code IDs and their names are included in the category fields, formatted as `ID: Name`.

You can specify the `labels` parameter when calling the connector:

In [20]:
df = conn.data(cbs, "L_POSITION = 'D' AND L_REP_CTY = 'CH'", labels="name")

print(df)

                                             FREQ  \
SERIES_KEY                 TIME_PERIOD              
Q.S.CH.4B.U.D.A.A.TO1.A.3P 2005-Q2      Quarterly   
                           2005-Q3      Quarterly   
                           2005-Q4      Quarterly   
                           2006-Q1      Quarterly   
                           2006-Q2      Quarterly   
...                                           ...   
Q.S.CH.4R.U.D.A.A.TO1.A.5J 2025-Q2      Quarterly   
                           2025-Q3      Quarterly   
                           2025-Q4      Quarterly   
                           2026-Q1      Quarterly   
                           2026-Q2      Quarterly   

                                                           L_MEASURE  \
SERIES_KEY                 TIME_PERIOD                                 
Q.S.CH.4B.U.D.A.A.TO1.A.3P 2005-Q2      Amounts outstanding / Stocks   
                           2005-Q3      Amounts outstanding / Stocks   
                      

### Selecting Specific Columns

The `columns` parameter allows you to specify which columns should be included in the resulting DataFrame. This is useful when you are only interested in a subset of the components, as it can reduce the size of the DataFrame and make the output more focused.

In [21]:
df = conn.data(
    cbs,
    "L_POSITION = 'D' AND L_REP_CTY = 'CH'",
    columns=["OBS_VALUE", "OBS_STATUS"]
)

print(df)

    TIME_PERIOD                  SERIES_KEY OBS_STATUS   OBS_VALUE
0       2005-Q2  Q.S.CH.4B.U.D.A.A.TO1.A.3P          A    805455.0
1       2005-Q3  Q.S.CH.4B.U.D.A.A.TO1.A.3P          A    847949.0
2       2005-Q4  Q.S.CH.4B.U.D.A.A.TO1.A.3P          A    794753.0
3       2006-Q1  Q.S.CH.4B.U.D.A.A.TO1.A.3P          A    919947.0
4       2006-Q2  Q.S.CH.4B.U.D.A.A.TO1.A.3P          A    989307.0
..          ...                         ...        ...         ...
165     2025-Q2  Q.S.CH.4R.U.D.A.A.TO1.A.5J          A  560048.557
166     2025-Q3  Q.S.CH.4R.U.D.A.A.TO1.A.5J          A  571884.474
167     2025-Q4  Q.S.CH.4R.U.D.A.A.TO1.A.5J          A   608926.95
168     2026-Q1  Q.S.CH.4R.U.D.A.A.TO1.A.5J          A  621052.768
169     2026-Q2  Q.S.CH.4R.U.D.A.A.TO1.A.5J          A   633359.74

[170 rows x 4 columns]


If the `columns` parameter is not specified, all components defined in the Data Structure Definition (DSD) are included in the DataFrame.